#BDMI Final Project - Financial - Elias Jr

In [ ]:
# Clone the official repository
!git clone https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory

# Install the package along with training metrics dependencies
!pip install -e .[metrics]
# Install bitsandbytes for optimized memory management
!pip install bitsandbytes

Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 27198, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 27198 (delta 13), reused 2 (delta 2), pack-reused 27161 (from 3)
Receiving objects: 100% (27198/27198), 13.14 MiB | 13.47 MiB/s, done.
Resolving deltas: 100% (19505/19505), done.
/content/LLaMA-Factory
Obtaining file:///content/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 46.7 MB/s eta 0:00:00
   

## Dataset creation e YAML Config

In [ ]:
import json

# 1. Properly register the multi-factor dataset inside LLaMA-Factory directory structure
dataset_info = {
  "financial_dpo_multi_factor": {
    "file_name": "large_scale_financial_dpo_(multi_factor).json",
    "ranking": True,
    "columns": {
      "prompt": "instruction",
      "query": "input",
      "chosen": "chosen",
      "rejected": "rejected"
    }
  }
}

with open("data/dataset_info.json", "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, indent=2, ensure_ascii=False)

# 2. Generate the fully commented English training specification file
yaml_content = """
### model
model_name_or_path: meta-llama/Meta-Llama-3-8B-Instruct

### method
stage: dpo
do_train: true
finetuning_type: lora
lora_target: all
pref_beta: 0.1
pref_loss: sigmoid

### dataset
dataset: financial_dpo_multi_factor
template: llama3
cutoff_len: 1024
max_samples: 1000
overwrite_cache: true
preprocessing_num_workers: 16

### output
output_dir: saves/llama3-8b/lora/dpo_financial
logging_steps: 10
save_steps: 100
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 2
gradient_accumulation_steps: 8
learning_rate: 5.0e-6
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 18000000
"""

with open("dpo_config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_content.strip())

print("Arquivo YAML atualizado com o nome correto do modelo!")

Arquivo YAML atualizado com o nome correto do modelo!


In [ ]:
#*****************************************

from huggingface_hub import notebook_login

notebook_login()

## Model Qwen2.5-7B

In [ ]:
yaml_content = """
### model
model_name_or_path: Qwen/Qwen2.5-7B-Instruct

### method
stage: dpo
do_train: true
finetuning_type: lora
lora_target: all
pref_beta: 0.1
pref_loss: sigmoid

### dataset
dataset: financial_dpo_multi_factor
template: qwen
cutoff_len: 1024
max_samples: 1000
overwrite_cache: true
preprocessing_num_workers: 16

### output
output_dir: saves/qwen-7b/lora/dpo_financial
logging_steps: 10
save_steps: 100
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 2
gradient_accumulation_steps: 8
learning_rate: 5.0e-6
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 18000000
"""

with open("qwen2_5dpo_config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_content.strip())

print("Arquivo YAML atualizado para usar o Qwen 2.5 (Livre de bloqueios)!")

Arquivo YAML atualizado para usar o Qwen 2.5 (Livre de bloqueios)!


In [ ]:
!llamafactory-cli train qwen2_5dpo_config.yaml

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|2026-05-17 01:48:52] llamafactory.hparams.parser:507 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
config.json: 100% 663/663 [00:00<00:00, 2.75MB/s]
[INFO|configuration_utils.py:667] 2026-05-17 01:48:52,537 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28/config.json
[INFO|configuration_utils.py:739] 2026-05-17 01:48:52,541 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 3584,
  "initializer_range": 0.02,
  "intermediate_size": 18944,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    

## Model Meta-Llama-3-8B

In [ ]:
yaml_content = """
### model
model_name_or_path: meta-llama/Meta-Llama-3-8B-Instruct  # Path to Meta's Llama 3 8B Instruct base model

### method
stage: dpo  # Specifies the training stage as Direct Preference Optimization
do_train: true  # Enables the training phase execution
finetuning_type: lora  # Uses Low-Rank Adaptation (LoRA) for parameter-efficient fine-tuning
lora_target: all  # Applies LoRA weights to all available linear layers to maximize capacity
pref_beta: 0.1  # KL divergence penalty coefficient; controls alignment strictness
pref_loss: sigmoid  # Loss function type used for evaluating preference ranking alignment

### dataset
dataset: financial_dpo_multi_factor  # Target multi-factor financial dataset registered in dataset_info.json
template: llama3  # Prompt template styling specifically matching Llama 3 tokenization requirements
cutoff_len: 1024  # Maximum sequence length (tokens) allowed before truncation
max_samples: 1000  # Upper limit of training samples to load from the JSON file
overwrite_cache: true  # Forces rewriting the preprocessed tokenization cache to avoid stale data
preprocessing_num_workers: 16  # Multi-processing CPU workers utilized during initial data tokenization

### output
output_dir: saves/llama3-8b/lora/dpo_financial  # Destination directory path for final weights and checkpoints (completely isolated)
logging_steps: 10  # Frequency of training steps before telemetry metrics are written
save_steps: 100  # Interval of training steps before saving a model checkpoint state
plot_loss: true  # Automatically plots and saves the loss curve diagram in the output directory
overwrite_output_dir: true  # Overwrites pre-existing target folders to clear previous run remnants

### train
per_device_train_batch_size: 2  # Training batch allocation per active GPU node
gradient_accumulation_steps: 8  # Steps over which gradients accumulate before an optimizer update (Effective Batch Size = 16)
learning_rate: 5.0e-6  # Peak learning rate for the AdamW optimization function
num_train_epochs: 3.0  # Number of complete tracking cycles through the entire dataset
lr_scheduler_type: cosine  # Decay curve style controlling the optimization pace over time
warmup_ratio: 0.1  # Fraction of the overall execution spent ramping up the learning rate
bf16: true  # Employs Brain Floating Point 16-bit precision for high-performance tensor acceleration
ddp_timeout: 18000000  # Extended multi-node sync timeout duration for Distributed Data Parallel arrays
"""

with open("llama3_dpo_config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_content.strip())

print("Arquivo 'llama3_dpo_config.yaml' criado com sucesso e isolado!")

Arquivo 'llama3_dpo_config.yaml' criado com sucesso e isolado!


In [ ]:
!llamafactory-cli train llama3_dpo_config.yaml

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|2026-05-17 02:08:25] llamafactory.hparams.parser:507 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
config.json: 100% 654/654 [00:00<00:00, 2.72MB/s]
[INFO|configuration_utils.py:667] 2026-05-17 02:08:25,499 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--meta-llama--Meta-Llama-3-8B-Instruct/snapshots/8afb486c1db24fe5011ec46dfbe5b5dccdb575c2/config.json
[INFO|configuration_utils.py:739] 2026-05-17 02:08:25,502 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": 128009,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 8192,
  "mlp_bias": false

## Modelo Gemma 2

In [ ]:
yaml_content = """
### model
model_name_or_path: google/gemma-2-9b-it

### method
stage: dpo
do_train: true
finetuning_type: lora
lora_target: all
pref_beta: 0.1
pref_loss: sigmoid

### dataset
dataset: financial_dpo_multi_factor
template: gemma
cutoff_len: 1024
max_samples: 1000
overwrite_cache: true
preprocessing_num_workers: 16

### output
output_dir: saves/gemma2-9b/lora/dpo_financial
logging_steps: 10
save_steps: 100
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 2
gradient_accumulation_steps: 8
learning_rate: 5.0e-6
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 18000000
"""

with open("gemma2_dpo_config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_content.strip())

print("Configuração do Gemma 2 criada!")

Configuração do Gemma 2 criada!


In [ ]:
!llamafactory-cli train gemma2_dpo_config.yaml

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|2026-05-17 02:35:45] llamafactory.hparams.parser:507 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
config.json: 100% 857/857 [00:00<00:00, 3.17MB/s]
[INFO|configuration_utils.py:771] 2026-05-17 02:35:45,742 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--google--gemma-2-9b-it/snapshots/11c9b309abf73637e4b6f9a3fa1e92e615547819/config.json
[INFO|configuration_utils.py:847] 2026-05-17 02:35:45,748 >> Model config Gemma2Config {
  "architectures": [
    "Gemma2ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "attn_logit_softcapping": 50.0,
  "bos_token_id": 2,
  "cache_implementation": "hybrid",
  "dtype": "bfloat16",
  "eos_token_id": 1,
  "final_logit_softcapping": 30.0,
  "head_dim": 256,
  "hidden_act": "gelu_pytorch_tanh",
  "hidden_activation": "ge

## Modelo DeepSeek-R1-Distill-Qwen-7B

In [ ]:
yaml_content = """
### model
model_name_or_path: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B

### method
stage: dpo
do_train: true
finetuning_type: lora
lora_target: all
pref_beta: 0.1
pref_loss: sigmoid

### dataset
dataset: financial_dpo_multi_factor
template: qwen  # Os modelos destilados do Qwen utilizam o template nativo do Qwen
cutoff_len: 1024
max_samples: 1000
overwrite_cache: true
preprocessing_num_workers: 16

### output
output_dir: saves/deepseek-r1-7b/lora/dpo_financial  # Pasta totalmente isolada
logging_steps: 10
save_steps: 100
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 2
gradient_accumulation_steps: 8
learning_rate: 5.0e-6
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 18000000
"""

with open("deepseek_dpo_config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_content.strip())

print("Configuração do DeepSeek-R1-Distill-Qwen-7B criada com sucesso!")

Configuração do DeepSeek-R1-Distill-Qwen-7B criada com sucesso!


In [ ]:
!llamafactory-cli train deepseek_dpo_config.yaml

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|2026-05-17 02:59:00] llamafactory.hparams.parser:507 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
config.json: 100% 680/680 [00:00<00:00, 3.07MB/s]
[INFO|configuration_utils.py:771] 2026-05-17 02:59:01,167 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--deepseek-ai--DeepSeek-R1-Distill-Qwen-7B/snapshots/916b56a44061fd5cd7d6a8fb632557ed4f724f60/config.json
[INFO|configuration_utils.py:847] 2026-05-17 02:59:01,172 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 3584,
  "initializer_range": 0.02,
  "intermediate_size": 18944,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attentio

# Chat Models

In [ ]:
yaml_content = """
model_name_or_path: meta-llama/Meta-Llama-3-8B-Instruct
adapter_name_or_path: saves/llama3-8b/lora/dpo_financial
template: llama3
finetuning_type: lora
"""

with open("llama3_infer.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_content.strip())

print("Arquivo de inferência do Llama 3 criado!")

Arquivo de inferência do Llama 3 criado!


In [ ]:
!llamafactory-cli chat llama3_infer.yaml

[INFO|configuration_utils.py:771] 2026-05-17 03:16:14,083 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--meta-llama--Meta-Llama-3-8B-Instruct/snapshots/8afb486c1db24fe5011ec46dfbe5b5dccdb575c2/config.json
[INFO|configuration_utils.py:847] 2026-05-17 03:16:14,088 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": 128009,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 8192,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "rope_theta": 500000.0,
    "rope_type": "default"
  },
  "tie_word_embeddings": false,
  "tr

## Download Weights

In [ ]:
# Compacta toda a estrutura de salvamento dos 3 modelos em um único arquivo zip
!zip -r dpo_financial_experts.zip /content/LLaMA-Factory/saves/

  adding: content/LLaMA-Factory/saves/ (stored 0%)
  adding: content/LLaMA-Factory/saves/deepseek-r1-7b/ (stored 0%)
  adding: content/LLaMA-Factory/saves/deepseek-r1-7b/lora/ (stored 0%)
  adding: content/LLaMA-Factory/saves/deepseek-r1-7b/lora/dpo_financial/ (stored 0%)
  adding: content/LLaMA-Factory/saves/deepseek-r1-7b/lora/dpo_financial/training_rewards_accuracies.png (deflated 10%)
  adding: content/LLaMA-Factory/saves/deepseek-r1-7b/lora/dpo_financial/training_args.bin (deflated 53%)
  adding: content/LLaMA-Factory/saves/deepseek-r1-7b/lora/dpo_financial/all_results.json (deflated 38%)
  adding: content/LLaMA-Factory/saves/deepseek-r1-7b/lora/dpo_financial/checkpoint-100/ (stored 0%)
  adding: content/LLaMA-Factory/saves/deepseek-r1-7b/lora/dpo_financial/checkpoint-100/training_args.bin (deflated 53%)
  adding: content/LLaMA-Factory/saves/deepseek-r1-7b/lora/dpo_financial/checkpoint-100/rng_state.pth (deflated 27%)
  adding: content/LLaMA-Factory/saves/deepseek-r1-7b/lora/dpo_f

In [ ]:
from google.colab import files

files.download('dpo_financial_experts.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>